# VA-DCP — Colab Setup Only

Notebook ini menyiapkan tiga arm eksperimen: **A0 real-only**, **A1 real + naive copy-paste**, dan **A2 real + visibility-aware dense copy-paste**.

Notebook **tidak menjalankan training**. Cell terakhir hanya memverifikasi bahwa data siap dan berhenti pada status `TRAINING_READY`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

# Gunakan 'full' agar hasil benar-benar siap untuk screening training.
# Ubah ke 'smoke' hanya untuk uji cepat pipeline.
PROFILE = 'full'  # 'full' atau 'smoke'

if PROFILE == 'full':
    MAX_ASSETS_PER_CLASS = 500
    SYNTHETIC_IMAGES = 2000
else:
    MAX_ASSETS_PER_CLASS = 50
    SYNTHETIC_IMAGES = 20

SEED = 42
REPO_URL = 'https://github.com/ediprin/coffee-bean-detection.git'
BRANCH = 'agent/add-vadcp-pipeline'
REPO = Path('/content/coffee-bean-detection')
DATA_ROOT = Path('/content/coffee-defect-roboflow')
OBJECT_LIBRARY = Path(f'/content/coffee-object-library-{PROFILE}')
A1_ROOT = Path(f'/content/coffee-A1-naive-{PROFILE}')
A2_ROOT = Path(f'/content/coffee-A2-vadcp-{PROFILE}')
BACKGROUND_ROOT = None  # Isi Path('/content/backgrounds') jika ada foto background kosong.
DRIVE_ARTIFACTS = Path('/content/drive/MyDrive/coffee-bean-detection/vadcp-setup') / PROFILE
DRIVE_ARTIFACTS.mkdir(parents=True, exist_ok=True)

print('PROFILE               :', PROFILE)
print('MAX ASSETS PER CLASS  :', MAX_ASSETS_PER_CLASS)
print('SYNTHETIC IMAGES      :', SYNTHETIC_IMAGES)
print('DRIVE ARTIFACTS       :', DRIVE_ARTIFACTS)

In [ ]:
# Sinkronkan branch VA-DCP dan instal menggunakan Python kernel yang sama.
if REPO.is_dir():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)
src = str(REPO / 'src')
if src not in sys.path:
    sys.path.insert(0, src)

import coffee_detector
from coffee_detector.vadcp import CompositionSpec

print('IMPORT BERHASIL:', coffee_detector.__file__)
print('VA-DCP BERHASIL:', CompositionSpec)

In [ ]:
# Unduh dataset pilot Coffee Defect v11 jika belum ada di runtime.
if not (DATA_ROOT / 'data.yaml').is_file():
    from getpass import getpass
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'], check=True)
    from roboflow import Roboflow

    rf = Roboflow(api_key=getpass('Masukkan API key Roboflow: '))
    dataset = (
        rf.workspace('adrians')
        .project('coffee-defect-1eny4')
        .version(11)
        .download('yolov8', location=str(DATA_ROOT), overwrite=True)
    )
    DATA_ROOT = Path(dataset.location).resolve()

assert (DATA_ROOT / 'data.yaml').is_file(), f'data.yaml tidak ditemukan: {DATA_ROOT}'
print('DATASET SIAP:', DATA_ROOT)

In [ ]:
# Audit sumber nyata sebelum satu pun aset dipakai untuk sintesis.
from coffee_detector.audit_dataset import audit_dataset

source_audit_path = DRIVE_ARTIFACTS / 'source_dataset_audit.json'
source_audit = audit_dataset(DATA_ROOT, source_audit_path, near_threshold=-1)

print('Images:', source_audit['images_by_split'])
print('Boxes :', source_audit['boxes_by_split'])
print('Safe  :', source_audit['safe_for_training'])
assert source_audit['safe_for_training'], f'Periksa audit: {source_audit_path}'

In [ ]:
# Buat object library hanya dari bounding box split train.
library_manifest = OBJECT_LIBRARY / 'object_library.json'
if library_manifest.is_file():
    print('SKIP OBJECT LIBRARY: manifest ditemukan')
else:
    assert not OBJECT_LIBRARY.exists(), f'Output parsial, gunakan folder baru: {OBJECT_LIBRARY}'
    subprocess.run([
        sys.executable, '-u', '-m', 'coffee_detector.prepare_object_library',
        '--data-root', str(DATA_ROOT),
        '--output-root', str(OBJECT_LIBRARY),
        '--source-split', 'train',
        '--max-assets-per-class', str(MAX_ASSETS_PER_CLASS),
        '--seed', str(SEED),
    ], check=True)

library = json.loads(library_manifest.read_text(encoding='utf-8'))
print('OBJECT LIBRARY SIAP')
print('Assets by class:', library['audit']['assets_by_class'])
print('Failures       :', library['audit']['failures'])

In [ ]:
# Materialisasi A1 dan A2. Val/test selalu disalin dari data nyata.
def generate_arm(mode: str, output_root: Path) -> Path:
    marker = output_root / 'metadata' / 'generation_manifest.json'
    if marker.is_file():
        print(f'SKIP {mode}: manifest ditemukan')
        return marker
    assert not output_root.exists(), f'Output parsial, gunakan folder baru: {output_root}'
    command = [
        sys.executable, '-u', '-m', 'coffee_detector.generate_vadcp_dataset',
        '--real-data-root', str(DATA_ROOT),
        '--object-library', str(OBJECT_LIBRARY),
        '--output-root', str(output_root),
        '--mode', mode,
        '--synthetic-images', str(SYNTHETIC_IMAGES),
        '--objects-min', '12',
        '--objects-max', '30',
        '--minimum-visibility', '0.10',
        '--seed', str(SEED),
    ]
    if BACKGROUND_ROOT is not None:
        command.extend(['--background-root', str(BACKGROUND_ROOT)])
    subprocess.run(command, check=True)
    return marker

A1_MANIFEST = generate_arm('naive', A1_ROOT)
A2_MANIFEST = generate_arm('visibility', A2_ROOT)
print('A1 SIAP:', A1_MANIFEST)
print('A2 SIAP:', A2_MANIFEST)

In [ ]:
# Audit kedua arm sintetis.
from coffee_detector.audit_vadcp import audit_vadcp_dataset

A1_AUDIT_PATH = DRIVE_ARTIFACTS / 'A1_vadcp_audit.json'
A2_AUDIT_PATH = DRIVE_ARTIFACTS / 'A2_vadcp_audit.json'
A1_AUDIT = audit_vadcp_dataset(A1_ROOT, A1_AUDIT_PATH)
A2_AUDIT = audit_vadcp_dataset(A2_ROOT, A2_AUDIT_PATH)

print('A1 safe:', A1_AUDIT['safe_for_training'], A1_AUDIT['labeled_instances_by_visibility'])
print('A2 safe:', A2_AUDIT['safe_for_training'], A2_AUDIT['labeled_instances_by_visibility'])
assert A1_AUDIT['safe_for_training'], A1_AUDIT['errors'][:10]
assert A2_AUDIT['safe_for_training'], A2_AUDIT['errors'][:10]

In [ ]:
# Buat dan tampilkan visual audit A2 sebelum training diizinkan.
from coffee_detector.run_vadcp_visual_audit import run_vadcp_visual_audit
from IPython.display import display
from PIL import Image

visual_root = Path(f'/content/vadcp-visual-{PROFILE}')
visual = run_vadcp_visual_audit(A2_ROOT, visual_root, samples=20, seed=SEED)
contact_sheet = Path(visual['contact_sheet'])
drive_contact_sheet = DRIVE_ARTIFACTS / 'A2_contact_sheet.jpg'
shutil.copy2(contact_sheet, drive_contact_sheet)

display(Image.open(contact_sheet))
print('CONTACT SHEET:', drive_contact_sheet)

In [ ]:
# Final readiness gate. Cell ini TIDAK menjalankan training.
checks = {
    'A0 data.yaml': DATA_ROOT / 'data.yaml',
    'A1 data.yaml': A1_ROOT / 'data.yaml',
    'A2 data.yaml': A2_ROOT / 'data.yaml',
    'Object library': library_manifest,
    'A1 manifest': A1_MANIFEST,
    'A2 manifest': A2_MANIFEST,
    'A2 contact sheet': drive_contact_sheet,
}
missing = {name: str(path) for name, path in checks.items() if not Path(path).is_file()}
assert not missing, f'Artefak belum lengkap: {missing}'
assert A1_AUDIT['safe_for_training'] and A2_AUDIT['safe_for_training']

setup_summary = {
    'profile': PROFILE,
    'seed': SEED,
    'synthetic_images_per_arm': SYNTHETIC_IMAGES,
    'A0': str(DATA_ROOT),
    'A1': str(A1_ROOT),
    'A2': str(A2_ROOT),
    'training_executed': False,
    'training_ready': PROFILE == 'full',
}
summary_path = DRIVE_ARTIFACTS / 'setup_summary.json'
summary_path.write_text(json.dumps(setup_summary, indent=2), encoding='utf-8')

TRAINING_READY = PROFILE == 'full'
print('\n=== SETUP SELESAI ===')
print(json.dumps(setup_summary, indent=2))
print('SUMMARY:', summary_path)
print('TRAINING BELUM DIJALANKAN.')
print('TRAINING_READY =', TRAINING_READY)